In [ ]:
import gc
from pathlib import Path

import torch
import numpy as np
from diffusers.utils import export_to_video, load_image
# from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
# from transformers import CLIPVisionModel
from IPython import display

from wan import WanI2V
from wan.configs.wan_i2v_14B import i2v_14B



In [ ]:

# local_model_path = "../weights/Wan2.1-I2V-14B-480P-Diffusers"
# image_encoder = CLIPVisionModel.from_pretrained(local_model_path, subfolder="image_encoder", torch_dtype=torch.float32)
# vae = AutoencoderKLWan.from_pretrained(local_model_path, subfolder="vae", torch_dtype=torch.float32)

# pipe = WanImageToVideoPipeline.from_pretrained(
#     local_model_path,
#     vae=vae,
#     image_encoder=image_encoder,
#     torch_dtype=torch.bfloat16,
# )

# # pipe.enable_model_cpu_offload()
# # pipe.to("cuda")
# # pipe.text_encoder.to("cpu")


In [ ]:
wan_i2v = WanI2V(
    config=i2v_14B,
    checkpoint_dir="../weights/Wan2.1-I2V-14B-480P/",
    device_id=0,
    t5_cpu=True,
)

Loading checkpoint shards:   0%|          | 0/7 [00:00<?, ?it/s]

In [ ]:
NEGATIVE_PROMPT = "Bright tones, overexposed, static, blurred details, subtitles, style, works, paintings, images, static, overall gray, worst quality, low quality, JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, three legs, many people in the background, walking backwards"
MAX_AREA = 480 * 832

def generate_video(wan_i2v, prompt, image_path):
    image_path = Path(image_path)
    image = load_image(str(image_path))
    aspect_ratio = image.height / image.width

    mod_value = wan_i2v.config.vae_stride[1] * wan_i2v.config.patch_size[1]
    height = round(np.sqrt(MAX_AREA * aspect_ratio)) // mod_value * mod_value
    width = round(np.sqrt(MAX_AREA / aspect_ratio)) // mod_value * mod_value
    image = image.resize((width, height))

    torch.cuda.synchronize()
    gc.collect()
    torch.cuda.empty_cache()

    video = wan_i2v.generate(
        prompt,
        image,
        max_area=MAX_AREA,
        n_prompt=NEGATIVE_PROMPT,
        sampling_steps=10,
    )

    video = (video * 0.5 + 0.5).clamp(0, 1)
    # C T H W -> T H W C
    video = video.permute(1, 2, 3, 0).cpu().numpy()

    # video_path = image_path.with_suffix(".mp4")

    return export_to_video(video, fps=16)


In [ ]:
prompt = "The girl holding a laptop looks at the girl with glasses who is holding a phone."
image_path = "./images/three_people_interacting.jpg"

output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

In [ ]:
prompt = (
    "The girl holding a laptop looks at the guy in a blue shirt. "
    "The girl with glasses keeps looking at her phone."
)
image_path = "./images/three_people_interacting.jpg"

output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

In [ ]:
prompt = (
    "The girl holding a laptop first looks at the girl holding a phone. "
    "Then she looks at the guy in a blue shirt."
)
image_path = "./images/three_people_interacting.jpg"

output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

In [ ]:
prompt = (
    "The girl holding a laptop goes from looking at the girl holding a phone to looking at the guy in a blue shirt."
)
image_path = "./images/three_people_interacting.jpg"

output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

In [ ]:
prompt = (
    "The man first closes the book he's holding and puts it on the table. Then, he picks up the headphones sitting on the desk and wears them."
)
image_path = "./images/man_at_desk.jpg"

output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

100%|██████████| 10/10 [06:00<00:00, 36.01s/it]


In [ ]:
prompt = (
    "The bald man closes the book he's reading, stands up and walks away. The girl watering the plants puts the watering can on the ground."
)
image_path = "./images/girl_watering_plants_old_man_reading.png"

output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

100%|██████████| 10/10 [05:58<00:00, 35.84s/it]


In [ ]:
prompt = (
    "The girl watering the flowers turns around at looks at the man sitting on the bench. The man keeps reading his book."
)
image_path = "./images/girl_watering_plants_old_man_reading.png"

output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

[autoreload of wan.modules.custom_model failed: Traceback (most recent call last):
  File "/local_scratch/gzappavi/wan_experiments/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 325, in check
    superreload(m, reload, self.old_objects)
  File "/local_scratch/gzappavi/wan_experiments/.venv/lib/python3.11/site-packages/IPython/extensions/autoreload.py", line 580, in superreload
    module = reload(module)
             ^^^^^^^^^^^^^^
  File "/home/gzappavi/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 621, in _exec
  File "<frozen importlib._bootstrap_external>", line 936, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1074, in get_code
  File "<frozen importlib._bootstrap_external>", line 1004, in source_to_code
  File "<frozen importlib._bootstrap>", line 241, in _call_with_frames_removed
 

In [ ]:
prompt = (
    "The girl with blue hair stands up and leaves on her skateboard. The man with a long beard and a hat picks up his water bottle and drinks from it."
)
image_path = "./images/girl_texting_man_writing.png"
output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

100%|██████████| 10/10 [05:58<00:00, 35.86s/it]


In [ ]:
prompt = (
    "The girl with glasses covers her face with both hands, while the girl wearing a hat removes her hat with one hand, revealing her hair."
)
image_path = "./images/twin_girls.png"
output_video = generate_video(wan_i2v, prompt, image_path)
display.Video(output_video, embed=True)

100%|██████████| 10/10 [05:57<00:00, 35.77s/it]
